# 3 — Lateral spatial distribution

All-atom CHARMM36 binary bilayer (DOPC/DOPA, DOPC/DOPG, DOPC/DLPG, DOPC/DLPA).

2D number-density map of each lipid's phosphate in the membrane plane, per leaflet. One frame gives a sparse snapshot; a trajectory fills it in.


## Setup

In [ ]:
# !pip install "MDAnalysis>=2.4" numpy scipy matplotlib
%matplotlib inline
import os, re, itertools
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

## Config — edit these

You have **no trajectory**, so `TRAJ = None` loads the single `.gro`
frame (a **snapshot**). When you have an `.xtc`/`.trr`, set `TRAJ` to it
and every cell below averages over frames automatically.

In [ ]:
TOPOLOGY = "md.tpr"        # topology (has bonds + masses)
STRUCTURE= "md.gro"        # single-frame structure (.gro or .pdb)
TRAJ     = None            # e.g. "md_center.xtc" when you have one; None -> use STRUCTURE

LIPIDS   = ["DOPC", "DOPA"]        # the TWO lipids in THIS system
HEAD     = "P"                     # headgroup / leaflet marker
STEP     = 1                       # frame stride (only matters with a trajectory)
OUTDIR   = "results"; os.makedirs(OUTDIR, exist_ok=True)

## Load

In [ ]:
u = mda.Universe(TOPOLOGY, TRAJ) if TRAJ else mda.Universe(TOPOLOGY, STRUCTURE)
has_bonds = len(getattr(u, "bonds", [])) > 0
present   = [r for r in LIPIDS if len(u.select_atoms(f"resname {r}")) > 0]
nframes   = len(u.trajectory)
print(f"{len(u.atoms)} atoms, {nframes} frame(s), bonds={has_bonds}")
print("Lipids present:", present)
if nframes == 1:
    print("NOTE: single frame -> snapshot only (no time averaging).")

## 2D density per lipid, per leaflet

In [ ]:
BINS = 40
maps = {(r,l): np.zeros((BINS,BINS)) for r in present for l in ("upper","lower")}
cnt  = {k:0 for k in maps}
allP = u.select_atoms(f"name {HEAD} and resname " + " ".join(present))
for ts in u.trajectory[::STEP]:
    Lx,Ly = ts.dimensions[:2]; zmid = allP.positions[:,2].mean()
    ex = np.linspace(0,Lx,BINS+1); ey = np.linspace(0,Ly,BINS+1)
    for r in present:
        p = u.select_atoms(f"name {HEAD} and resname {r}").positions
        for l,m in (("upper",p[:,2]>=zmid),("lower",p[:,2]<zmid)):
            q=p[m]
            if len(q)==0: continue
            H,_,_=np.histogram2d(np.mod(q[:,0],Lx),np.mod(q[:,1],Ly),bins=[ex,ey])
            maps[(r,l)]+=H; cnt[(r,l)]+=1
keys=[k for k in maps if cnt[k]>0]; cols=min(4,len(keys)) or 1; rows=(len(keys)+cols-1)//cols
fig,ax=plt.subplots(rows,cols,figsize=(4*cols,3.4*rows),squeeze=False)
for a,(r,l) in zip(ax.flat,keys):
    im=a.imshow((maps[(r,l)]/cnt[(r,l)]).T,origin="lower",cmap="viridis")
    a.set_title(f"{r} {l}"); fig.colorbar(im,ax=a,shrink=.8)
for a in ax.flat[len(keys):]: a.axis("off")
plt.tight_layout(); plt.show()